# EKF-Adaptive Pruning: Stage 1 Baseline Training

**目标**: ResNet18-CIFAR baseline -> 93%+ test accuracy

**使用方式**: 从上到下按顺序运行每个cell即可。

第一次用建议先跑 `--epochs 5` 快速验证，再改 100。

## Step 1: 检查GPU

In [ ]:
!nvidia-smi

## Step 2: 挂载Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/ekf_adaptive_pruning'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f'[cwd] {os.getcwd()}')

## Step 3: 建立目录结构

In [ ]:
import os

os.makedirs('models', exist_ok=True)
os.makedirs('src', exist_ok=True)
open('models/__init__.py', 'w').close()
open('src/__init__.py', 'w').close()

print('目录创建完成')
print(os.listdir())

## Step 4: 写入 resnet18_cifar.py

用Python直接写入，避免%%writefile被截断。

In [ ]:
resnet_code = '''"""CIFAR-style ResNet-18."""
import torch
import torch.nn as nn
import torch.nn.functional as F


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(self.expansion * planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNet18CIFAR(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(BasicBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out


def resnet18_cifar(num_classes=10):
    return ResNet18CIFAR(num_classes=num_classes)
'''

with open('models/resnet18_cifar.py', 'w') as f:
    f.write(resnet_code)

print('写入完成:', os.path.getsize('models/resnet18_cifar.py'), '字节')

## Step 5: 写入 train_base.py

In [ ]:
train_code = '''"""ResNet18 CIFAR-10 baseline training."""
import os
import argparse
import time
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T

from models.resnet18_cifar import resnet18_cifar


def get_dataloaders(data_dir="./data", batch_size=128, num_workers=2):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)
    train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean, std)])
    test_tf = T.Compose([T.ToTensor(), T.Normalize(mean, std)])
    train_set = torchvision.datasets.CIFAR10(root=data_dir, train=True, download=True, transform=train_tf)
    test_set = torchvision.datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_tf)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n_batches = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch_size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=0.1)
    parser.add_argument("--momentum", type=float, default=0.9)
    parser.add_argument("--weight_decay", type=float, default=5e-4)
    parser.add_argument("--data_dir", type=str, default="./data")
    parser.add_argument("--save_path", type=str, default="./checkpoints/resnet18_cifar_base.pth")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[device] {device}")
    if device.type == "cuda":
        print(f"[gpu]    {torch.cuda.get_device_name(0)}")

    train_loader, test_loader = get_dataloaders(data_dir=args.data_dir, batch_size=args.batch_size)

    model = resnet18_cifar(num_classes=10).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[model]  ResNet18-CIFAR, params={n_params/1e6:.2f}M")

    optimizer = optim.SGD(model.parameters(), lr=args.lr, momentum=args.momentum, weight_decay=args.weight_decay, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    criterion = nn.CrossEntropyLoss()

    os.makedirs(os.path.dirname(args.save_path), exist_ok=True)
    best_acc = 0.0
    t_start = time.time()

    for epoch in range(1, args.epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        scheduler.step()
        test_acc = evaluate(model, test_loader, device)
        dt = time.time() - t0
        lr_now = optimizer.param_groups[0]["lr"]
        print(f"[ep {epoch:3d}/{args.epochs}] loss={train_loss:.4f}  test_acc={test_acc:.2f}%  lr={lr_now:.4f}  dt={dt:.1f}s")

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "test_acc": test_acc}, args.save_path)

    total_time = (time.time() - t_start) / 60
    print(f"\\n[done] best_acc={best_acc:.2f}%  total_time={total_time:.1f}min  saved={args.save_path}")


if __name__ == "__main__":
    main()
'''

with open('src/train_base.py', 'w') as f:
    f.write(train_code)

print('写入完成:', os.path.getsize('src/train_base.py'), '字节')

# 验证line 23
with open('src/train_base.py') as f:
    lines = f.readlines()
print(f'总行数: {len(lines)}')
print(f'line 23: {lines[22].rstrip()}')

## Step 6: Smoke test - 模型能不能正向传播

In [ ]:
import sys
sys.path.insert(0, '.')

for mod in list(sys.modules.keys()):
    if mod.startswith('models'):
        del sys.modules[mod]

import torch
from models.resnet18_cifar import resnet18_cifar

model = resnet18_cifar(num_classes=10).cuda()
x = torch.randn(4, 3, 32, 32).cuda()
y = model(x)
print(f'Input:  {x.shape}')
print(f'Output: {y.shape}')
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

## Step 7: 快速训练验证 (5 epochs, 约3-5分钟)

期待 5 epoch 后 test_acc 能到 60%+，说明 pipeline 稳了。

In [ ]:
!python src/train_base.py --epochs 5

## Step 8: 完整训练 (100 epochs, 约2-3小时)

快速验证通过之后，去掉下面的注释执行完整训练。

目标: best_acc 93%-94% 左右。

In [ ]:
# !python src/train_base.py --epochs 100